In [6]:
pip install dwave-ocean-sdk

Note: you may need to restart the kernel to use updated packages.


In [3]:
import dimod
from dwave.system import DWaveSampler
from dwave.samplers import SimulatedAnnealingSampler

usaremos dwave y ocean para resolver problemas QUBO

In [7]:
# Aquí indicas los coeficientes del Hamiltoniano de Ising que resuelve el problema
# Tomamos H1 = Z0 Z1 + Z0 Z2
J = {(0,1):1, (0,2):1} # define los coeficientes Jjk 
h = {} # define los coeficientes hj
# los coeficientes no especificados toman el valor 0 por defecto
problem = dimod.BinaryQuadraticModel(h, J, 0.0, dimod.BINARY)
# el offset toma el valor 0.0: termino constante que se añade al Hamiltoniano
#dimod.BINARY: Parametros binarios -> los valores de las variables son 0,1
print(f'The problem to solve is {problem}') 

The problem to solve is BinaryQuadraticModel({0: 0.0, 1: 0.0, 2: 0.0}, {(1, 0): 1.0, (2, 0): 1.0}, 0.0, 'BINARY')


In [8]:
sampler = SimulatedAnnealingSampler()
result = sampler.sample(problem, num_reads=100)

agg = result.aggregate() # aqui agrupo todos los resultados de las 100 lecturas
print(agg)

   0  1  2 energy num_oc.
0  0  0  1    0.0      18
1  1  0  0    0.0      27
2  0  1  1    0.0      17
3  0  0  0    0.0      16
4  0  1  0    0.0      22
['BINARY', 5 rows, 100 samples, 3 variables]


por lo que la clase BinaryQuadraticModel sirve para resolver problemas tanto de Ising como de QUBO

In [9]:
# Se intentara resolver el problema:
# MINIMIZAR        -5x0 + 3x1 - 2x2
# BAJO xj en {0,1}, j = 0,1,2; x0 + x2 <= 1; 3x0 -x1 + 3x2 <= 4
# que es una instancia de binary linear programming

In [10]:
# dimod tiene la clase: ConstrainedQuadraticModel que simpllifica el proceso de trabajar con problemas con condiciones restrictivas

x0 = dimod.Binary('x0')
x1 = dimod.Binary('x1')
x2 = dimod.Binary('x2')

# se han creado tres variables binarias y se les ha labeleado para poder usarlas

In [11]:
# ahora vamos a definir un objeto de ConstrainQuadraticModel y vamos a especificar el objetivo (funcion a minimizar) y fijar las restricciones

blp = dimod.ConstrainedQuadraticModel()
blp.set_objective(-5*x0+3*x1-2*x2)
blp.add_constraint(x0+x2<=1, 'First constraint')
blp.add_constraint(3*x0-x1+3*x2<=4, 'Second constraint')
print(f'variables: {blp.variables}')
print(f'objective: {blp.objective}')
print(f'constraints: {blp.constraints}')


variables: Variables(['x0', 'x1', 'x2'])
objective: ObjectiveView({'x0': -5.0, 'x1': 3.0, 'x2': -2.0}, {}, 0.0, {'x0': 'BINARY', 'x1': 'BINARY', 'x2': 'BINARY'})
constraints: {'First constraint': Le(ConstraintView({'x0': 1.0, 'x2': 1.0}, {}, 0.0, {'x0': 'BINARY', 'x2': 'BINARY'}), np.float64(1.0)), 'Second constraint': Le(ConstraintView({'x0': 3.0, 'x1': -1.0, 'x2': 3.0}, {}, 0.0, {'x0': 'BINARY', 'x1': 'BINARY', 'x2': 'BINARY'}), np.float64(4.0))}


La funcion objetivo y las condiciones restrictivas se presentan como funciones cuadratricas

Le: less than or equal to
tambien se pueden generar condiciones Ge: greater than
(Le + Ge) de la misma inecuacion = '='


YA HEMOS CONSTRUIDO EL PROBLEMA, AHORA VEREMOS COMO USAR EL PROBLEMA DEFINIDO PARA EVALUAR EL COSTO DE DIFERENTE ASIGNACION DE VALORES, COMPROBAR SI SATISFACEN LAS CONDICIONES Y ENCONTRAR LA SOLUCION OPTIMA

// RESOLVER CONSTRAINED QUADRATIC MODELS WITH DIMOD //

In [2]:
sample1 = {'x0': 1, 'x1': 1, 'x2': 1}
print(f'The assignment is {sample1}')
print(f'Its cost is {blp.objective.energy(sample1)}')
print(f'is it feasible? {blp.check_feasible(sample1)}')
print(f'The violations of the constraints are {blp.violations(sample1)}')

The assignment is {'x0': 1, 'x1': 1, 'x2': 1}


NameError: name 'blp' is not defined

esto nos indica que la asignacion no es viable, y nos da la cantidad por la que el lado izquierdo (donde las variables) de la desigualdad es mayor que el lado de los terminos independientes

In [13]:
sample2 = {'x0': 0, 'x1': 0, 'x2': 1}
print(f'The assignment is {sample2}')
print(f'Its cost is {blp.objective.energy(sample2)}')
print(f'is it feasible? {blp.check_feasible(sample2)}')
print(f'The violations of the constraints are {blp.violations(sample2)}')

The assignment is {'x0': 0, 'x1': 0, 'x2': 1}
Its cost is -2.0
is it feasible? True
The violations of the constraints are {'First constraint': np.float64(0.0), 'Second constraint': np.float64(-1.0)}


En este caso la asignacion si que es viable

In [14]:
# dimod tambien puede resolver por fuerza bruta

solver = dimod.ExactCQMSolver()
solution = solver.sample_cqm(blp)
print(f'The list of assignments is: \n {solution}')

The list of assignments is: 
   x0 x1 x2 energy num_oc. is_sat. is_fea.
6  1  0  1   -7.0       1 arra... np.F...
2  1  0  0   -5.0       1 arra... np.T...
7  1  1  1   -4.0       1 arra... np.F...
3  1  1  0   -2.0       1 arra... np.T...
4  0  0  1   -2.0       1 arra... np.T...
0  0  0  0    0.0       1 arra... np.T...
5  0  1  1    1.0       1 arra... np.T...
1  0  1  0    3.0       1 arra... np.T...
['INTEGER', 8 rows, 8 samples, 3 variables]


In [15]:
print(solution.first)
# obtenemos una solucion que no cumple las condiciones impuestas

Sample(sample={'x0': np.int64(1), 'x1': np.int64(0), 'x2': np.int64(1)}, energy=np.float64(-7.0), num_occurrences=np.int64(1), is_satisfied=array([False, False]), is_feasible=np.False_)


In [16]:
# se pueden eliminar las soluciones no viables
feasible_sol = solution.filter(lambda s: s.is_feasible)
print(feasible_sol)
print(feasible_sol.first)

  x0 x1 x2 energy num_oc. is_sat. is_fea.
2  1  0  0   -5.0       1 arra... np.T...
3  1  1  0   -2.0       1 arra... np.T...
4  0  0  1   -2.0       1 arra... np.T...
0  0  0  0    0.0       1 arra... np.T...
5  0  1  1    1.0       1 arra... np.T...
1  0  1  0    3.0       1 arra... np.T...
['INTEGER', 6 rows, 6 samples, 3 variables]
Sample(sample={'x0': np.int64(1), 'x1': np.int64(0), 'x2': np.int64(0)}, energy=np.float64(-5.0), num_occurrences=np.int64(1), is_satisfied=array([ True,  True]), is_feasible=np.True_)


todas estas computaciones se han realizado en un algoritmo clasico

Se pueden resolver probleas en annealers cuanticos

In [4]:
y0, y1 = dimod.Binaries(['y0', 'y1'])
cqm = dimod.ConstrainedQuadraticModel()
cqm.set_objective(-2*y0-3*y1)
cqm.add_constraint(y0+2*y1<=2)

#podemos transformar este problema con condiciones en otro sin condiciones de la siguiente forma:
qubo, invert = dimod.cqm_to_bqm(cqm, lagrange_multiplier=5) # lagrange_multiplier es la variable de coste
print(qubo)

BinaryQuadraticModel({'y0': -17.0, 'y1': -23.0, 'slack_vbc43728b5099487088e97a393e9f39df_0': -15.0, 'slack_vbc43728b5099487088e97a393e9f39df_1': -15.0}, {('y1', 'y0'): 20.0, ('slack_vbc43728b5099487088e97a393e9f39df_0', 'y0'): 10.0, ('slack_vbc43728b5099487088e97a393e9f39df_0', 'y1'): 20.0, ('slack_vbc43728b5099487088e97a393e9f39df_1', 'y0'): 10.0, ('slack_vbc43728b5099487088e97a393e9f39df_1', 'y1'): 20.0, ('slack_vbc43728b5099487088e97a393e9f39df_1', 'slack_vbc43728b5099487088e97a393e9f39df_0'): 10.0}, 20.0, 'BINARY')


In [5]:
# BinaryQuadraticModel({'y0': -17.0, 'y1': -23.0, 
# 'slack_v986cf994b43440ebb5e9d864132d05e0_0': -15.0, 
# 'slack_v986cf994b43440ebb5e9d864132d05e0_1': -15.0}, 
# {('y1', 'y0'): 20.0,
#  ('slack_v986cf994b43440ebb5e9d864132d05e0_0', 'y0'): 10.0, 
# ('slack_v986cf994b43440ebb5e9d864132d05e0_0', 'y1'): 20.0, 
# ('slack_v986cf994b43440ebb5e9d864132d05e0_1', 'y0'): 10.0, 
# ('slack_v986cf994b43440ebb5e9d864132d05e0_1', 'y1'): 20.0, 
# ('slack_v986cf994b43440ebb5e9d864132d05e0_1', 'slack_v986cf994b43440ebb5e9d864132d05e0_0'): 10.0}, 20.0, 'BINARY')

In [8]:
sampler = SimulatedAnnealingSampler()
result = sampler.sample(qubo, num_reads=100)
print(result)

   slack_vbc43728b5099487088e97a393e9f39df_0 ... y1 energy num_oc.
0                                          0 ...  1   -3.0       1
1                                          0 ...  1   -3.0       1
2                                          0 ...  1   -3.0       1
3                                          0 ...  1   -3.0       1
4                                          0 ...  1   -3.0       1
5                                          0 ...  1   -3.0       1
6                                          0 ...  1   -3.0       1
7                                          0 ...  1   -3.0       1
8                                          0 ...  1   -3.0       1
9                                          0 ...  1   -3.0       1
10                                         0 ...  1   -3.0       1
12                                         0 ...  1   -3.0       1
13                                         0 ...  1   -3.0       1
15                                         0 ...  1   -3.0    

In [9]:
agg = result.aggregate() # aquí agrupo todos los resultados de las 100 lecturas
print(agg)
print(agg.first)

  slack_vbc43728b5099487088e97a393e9f39df_0 ... y1 energy num_oc.
0                                         0 ...  1   -3.0      56
1                                         1 ...  0   -2.0      23
2                                         0 ...  0   -2.0      21
['BINARY', 3 rows, 100 samples, 4 variables]
Sample(sample={'slack_vbc43728b5099487088e97a393e9f39df_0': np.int8(0), 'slack_vbc43728b5099487088e97a393e9f39df_1': np.int8(0), 'y0': np.int8(0), 'y1': np.int8(1)}, energy=np.float64(-3.0), num_occurrences=np.int64(56))


In [10]:
# Hay exceso de informacion en los resultados
# vamos a usar ahora el objeto 'invert'
samples = []
occurrences = []

for s in result.data():
    samples.append(invert(s.sample))
    # se usa invert para eliminar las variables de slack
    occurrences.append(s.num_occurrences)
samplest = dimod.SampleSet.from_samples_cqm(samples,cqm,num_ocurrences=occurrences).aggregate() # añadirmos aggregate para que no sea too much information

print(f'The solution to the original problem are: \n {samplest}')

The solution to the original problem are: 
   y0 y1 energy num_oc. num_oc. is_sat. is_fea.
0  0  1   -3.0      56       1 arra... np.T...
1  1  0   -2.0      44       1 arra... np.T...
['INTEGER', 2 rows, 100 samples, 2 variables]
